# Complete Model Training Notebook for Dissertation

This notebook trains **ALL models** in the codebase and outputs comprehensive metrics and visualizations.

## Models Covered (21 Total)
1. **Baseline Models (5)**: LSTM, GRU, BiLSTM, Attention LSTM, Transformer
2. **PINN Models (6)**: Baseline PINN, GBM, OU, Black-Scholes, GBM+OU, Global
3. **Advanced PINN (3)**: StackedPINN, ResidualPINN, SpectralPINN
4. **Volatility Models (6)**: Vol-LSTM, Vol-GRU, Vol-Transformer, Vol-PINN, Heston-PINN, Stacked-Vol-PINN

## Setup Instructions
1. Use GPU runtime: Runtime -> Change runtime type -> GPU
2. Clone or upload the repo to Colab
3. Run all cells in order

In [3]:
#@title 1. Environment Setup
import os
import sys
import subprocess
from pathlib import Path

# Detect environment
# Check if truly running in Colab's cloud environment vs Colab kernel in VS Code
IN_COLAB_CLOUD = 'google.colab' in sys.modules and os.path.exists('/content')
IN_VSCODE_WITH_COLAB = 'google.colab' in sys.modules and not os.path.exists('/content')
IN_LOCAL = not ('google.colab' in sys.modules)

if IN_COLAB_CLOUD:
    print('Environment: Google Colab (cloud)')
elif IN_VSCODE_WITH_COLAB:
    print('Environment: VS Code with Colab kernel (local files)')
else:
    print('Environment: Local Python')

# Set PROJECT_ROOT based on environment
if IN_COLAB_CLOUD:
    # Running in actual Google Colab cloud - need to mount Drive or clone repo
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/Dissertaion-Project'
    
    if not os.path.exists(PROJECT_ROOT):
        raise FileNotFoundError(f'Project not found at {PROJECT_ROOT}. Upload to Google Drive first.')
else:
    # Running locally (VS Code with Colab kernel OR regular local Python)
    # The notebook is in Jupyter/ subdirectory
    notebook_dir = Path(os.getcwd())
    
    # Try to find project root
    possible_roots = [
        Path('/Users/mustif/Documents/GitHub/Dissertaion-Project'),  # Your explicit path
        notebook_dir.parent,  # If cwd is Jupyter/
        notebook_dir,          # If cwd is project root
    ]
    
    PROJECT_ROOT = None
    for root in possible_roots:
        if root.exists() and (root / 'src').exists() and (root / 'backend').exists():
            PROJECT_ROOT = str(root)
            break
    
    if PROJECT_ROOT is None:
        PROJECT_ROOT = '/Users/mustif/Documents/GitHub/Dissertaion-Project'
        print(f'Warning: Using hardcoded path: {PROJECT_ROOT}')

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

# Verify project structure
required_dirs = ['src', 'backend', 'src/models', 'src/training']
for d in required_dirs:
    path = os.path.join(PROJECT_ROOT, d)
    if os.path.exists(path):
        print(f'  ✓ {d}/')
    else:
        print(f'  ✗ {d}/ MISSING')

# Check GPU availability
try:
    import torch
    if torch.cuda.is_available():
        print(f'GPU: {torch.cuda.get_device_name(0)}')
    elif torch.backends.mps.is_available():
        print('GPU: Apple MPS (Metal)')
    else:
        print('GPU: Not available (using CPU)')
except ImportError:
    print('PyTorch not imported yet')

# Install dependencies only in actual Colab cloud
if IN_COLAB_CLOUD:
    print('Installing dependencies...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 
                    'torch', 'numpy', 'pandas', 'matplotlib', 'seaborn', 
                    'scikit-learn', 'tqdm', 'yfinance', 'python-dotenv', 'pydantic'], check=True)
    
    req_file = Path(PROJECT_ROOT) / 'backend' / 'requirements.txt'
    if req_file.exists():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_file)], check=False)
    print('Dependencies installed!')

print('Setup complete!')

In [ ]:
#@title 2. Import Core Modules
import json
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

try:
    from src.models.model_registry import ModelRegistry
    from src.training.trainer import Trainer
    from src.data.fetcher import DataFetcher
    from src.data.preprocessor import DataPreprocessor
    from src.data.dataset import PhysicsAwareDataset, collate_fn_with_metadata
    from src.utils.config import get_config, get_research_config
    from src.utils.reproducibility import set_seed, get_device
    from src.evaluation.metrics import calculate_metrics
    from src.evaluation.financial_metrics import calculate_financial_metrics, compute_strategy_returns
    from src.training.curriculum_scheduler import CurriculumScheduler
    HAS_SRC = True
    print('Successfully imported src modules - using REAL neural network training')
except ImportError as e:
    HAS_SRC = False
    print(f'Failed to import src modules: {e}')
    raise

# Import volatility trainer
try:
    from src.training.volatility_trainer import VolatilityDataPreparer, VolatilityTrainer
    HAS_VOL_TRAINER = True
    print('Volatility trainer imported successfully')
except ImportError as e:
    HAS_VOL_TRAINER = False
    print(f'Volatility trainer not available: {e}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

set_seed(42)
print('Random seed set to 42')

Failed to import src modules: No module named 'src'


ModuleNotFoundError: No module named 'src'

In [ ]:
#@title 3. Define Model Groups (ALL 21 MODELS)
project_root = Path(PROJECT_ROOT)
registry = ModelRegistry(project_root)
research_cfg = get_research_config()

# Baseline Models (5)
BASELINE_MODELS = ['lstm', 'gru', 'bilstm', 'attention_lstm', 'transformer']

# PINN Models (6)
PINN_MODELS = ['baseline_pinn', 'gbm', 'ou', 'black_scholes', 'gbm_ou', 'global']

# Advanced PINN Models (3) - Including Spectral
ADVANCED_PINN_MODELS = ['stacked', 'residual', 'spectral_pinn']

# Volatility Models (6)
VOLATILITY_MODELS = ['vol_lstm', 'vol_gru', 'vol_transformer', 'vol_pinn', 'heston_pinn', 'stacked_vol_pinn']

# All price prediction models
PRICE_MODELS = BASELINE_MODELS + PINN_MODELS + ADVANCED_PINN_MODELS

# Total model count
TOTAL_MODELS = len(PRICE_MODELS) + len(VOLATILITY_MODELS)

print(f'ModelRegistry contains {len(registry.models)} model definitions')
print(f'\n=== MODELS TO TRAIN ({TOTAL_MODELS} total) ===')
print(f'\nBaseline models ({len(BASELINE_MODELS)}): {BASELINE_MODELS}')
print(f'PINN models ({len(PINN_MODELS)}): {PINN_MODELS}')
print(f'Advanced PINN ({len(ADVANCED_PINN_MODELS)}): {ADVANCED_PINN_MODELS}')
print(f'Volatility models ({len(VOLATILITY_MODELS)}): {VOLATILITY_MODELS}')
print(f'\nResearch config: epochs={research_cfg.epochs}, batch_size={research_cfg.batch_size}')

In [ ]:
#@title 4. Prepare Price Prediction Training Data
def prepare_training_data(ticker='^GSPC', use_multi_ticker=True):
    config = get_config()
    research_cfg = get_research_config()
    
    fetcher = DataFetcher()
    preprocessor = DataPreprocessor()
    
    tickers = config.data.tickers[:10] if use_multi_ticker else [ticker]
    print(f'Fetching data for {len(tickers)} tickers: {tickers[:5]}...')
    
    df = fetcher.fetch_and_store(
        tickers=tickers,
        start_date=config.data.start_date,
        end_date=config.data.end_date,
        force_refresh=False
    )
    
    if df.empty:
        raise ValueError('No data fetched!')
    
    df = preprocessor.process_and_store(df)
    
    feature_cols = [
        'close', 'volume', 'log_return', 'simple_return',
        'rolling_volatility_5', 'rolling_volatility_20',
        'momentum_5', 'momentum_20', 'rsi_14', 'macd', 'macd_signal'
    ]
    feature_cols = [col for col in feature_cols if col in df.columns]
    print(f'Using {len(feature_cols)} features')
    
    train_df, val_df, test_df = preprocessor.split_temporal(df)
    
    for col in feature_cols:
        train_df[col] = train_df[col].astype(np.float64)
        val_df[col] = val_df[col].astype(np.float64)
        test_df[col] = test_df[col].astype(np.float64)
    
    train_df_norm, scalers = preprocessor.normalize_features(train_df, feature_cols, method='standard')
    
    val_df_norm = val_df.copy()
    test_df_norm = test_df.copy()
    
    for ticker_name in val_df['ticker'].unique():
        if ticker_name in scalers:
            val_mask = val_df_norm['ticker'] == ticker_name
            val_df_norm.loc[val_mask, feature_cols] = scalers[ticker_name].transform(
                val_df_norm.loc[val_mask, feature_cols])
    
    for ticker_name in test_df['ticker'].unique():
        if ticker_name in scalers:
            test_mask = test_df_norm['ticker'] == ticker_name
            test_df_norm.loc[test_mask, feature_cols] = scalers[ticker_name].transform(
                test_df_norm.loc[test_mask, feature_cols])
    
    seq_length = research_cfg.sequence_length
    target_col = 'log_return'
    
    X_train, y_train, tickers_train = preprocessor.create_sequences(
        train_df_norm, feature_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)
    
    X_val, y_val, tickers_val = preprocessor.create_sequences(
        val_df_norm, feature_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)
    
    X_test, y_test, tickers_test = preprocessor.create_sequences(
        test_df_norm, feature_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)
    
    physics_cols = ['close', 'log_return', 'rolling_volatility_20']
    physics_cols = [c for c in physics_cols if c in df.columns]
    
    P_train, _, _ = preprocessor.create_sequences(
        train_df, physics_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)
    P_val, _, _ = preprocessor.create_sequences(
        val_df, physics_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)
    P_test, _, _ = preprocessor.create_sequences(
        test_df, physics_cols, target_col=target_col,
        sequence_length=seq_length, forecast_horizon=1)
    
    train_ds = PhysicsAwareDataset(
        X_train, y_train, tickers_train,
        prices=P_train[:, :, 0], returns=P_train[:, :, 1],
        volatilities=P_train[:, :, 2] if P_train.shape[2] > 2 else None)
    
    val_ds = PhysicsAwareDataset(
        X_val, y_val, tickers_val,
        prices=P_val[:, :, 0], returns=P_val[:, :, 1],
        volatilities=P_val[:, :, 2] if P_val.shape[2] > 2 else None)
    
    test_ds = PhysicsAwareDataset(
        X_test, y_test, tickers_test,
        prices=P_test[:, :, 0], returns=P_test[:, :, 1],
        volatilities=P_test[:, :, 2] if P_test.shape[2] > 2 else None)
    
    input_dim = X_train.shape[2]
    
    print(f'Dataset sizes: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')
    print(f'Input dimension: {input_dim}')
    print(f'Sequence length: {seq_length}')
    
    return train_ds, val_ds, test_ds, input_dim, scalers

train_ds, val_ds, test_ds, input_dim, scalers = prepare_training_data()

In [ ]:
#@title 5. Prepare Volatility Training Data
def prepare_volatility_data():
    """Prepare data specifically for volatility forecasting models."""
    if not HAS_VOL_TRAINER:
        print('Volatility trainer not available - skipping volatility data prep')
        return None
    
    config = get_config()
    fetcher = DataFetcher()
    preprocessor = DataPreprocessor()
    
    print('Fetching data for volatility models...')
    vol_df = fetcher.fetch_and_store(
        tickers=['^GSPC'],
        start_date=config.data.start_date,
        end_date=config.data.end_date,
        force_refresh=False
    )
    vol_df = preprocessor.process_and_store(vol_df)
    single = vol_df[vol_df['ticker'] == '^GSPC'].sort_values('time').set_index('time')
    
    # Prepare volatility features
    preparer = VolatilityDataPreparer(seq_length=40, horizon=5)
    features = preparer.compute_volatility_features(single)
    
    # Drop NaN rows
    features = features.dropna()
    aligned_returns = single.loc[features.index, 'log_return']
    
    dataset = preparer.prepare(
        features.values, 
        aligned_returns.values, 
        dates=features.index.values
    )
    
    print(f'Volatility dataset: Train={dataset.X_train.shape[0]}, Val={dataset.X_val.shape[0]}')
    print(f'Volatility features: {dataset.X_train.shape[2]}')
    
    return dataset

vol_dataset = prepare_volatility_data()

In [ ]:
#@title 6. Training Helper Functions
train_loader = DataLoader(train_ds, batch_size=research_cfg.batch_size, 
                          shuffle=True, collate_fn=collate_fn_with_metadata)
val_loader = DataLoader(val_ds, batch_size=research_cfg.batch_size, 
                        shuffle=False, collate_fn=collate_fn_with_metadata)
test_loader = DataLoader(test_ds, batch_size=research_cfg.batch_size, 
                         shuffle=False, collate_fn=collate_fn_with_metadata)

results_dir = project_root / 'results' / 'colab_runs'
results_dir.mkdir(parents=True, exist_ok=True)

def evaluate_model(model, loader, model_name):
    model.eval()
    predictions, targets = [], []
    
    with torch.no_grad():
        for batch in loader:
            sequences, target, metadata = batch
            sequences = sequences.to(device)
            target = target.to(device)
            
            output = model(sequences)
            if isinstance(output, tuple):
                output = output[0]
            
            predictions.append(output.cpu().numpy().squeeze())
            targets.append(target.cpu().numpy().squeeze())
    
    predictions = np.concatenate(predictions)
    targets = np.concatenate(targets)
    
    ml_metrics = calculate_metrics(targets, predictions, prefix='')
    
    try:
        strategy_returns = compute_strategy_returns(
            predictions=predictions,
            actual_prices=targets,
            are_returns=True,
            transaction_cost=0.001
        )
        fin_metrics = calculate_financial_metrics(
            returns=strategy_returns,
            risk_free_rate=0.02,
            periods_per_year=252,
            prefix=''
        )
    except Exception as e:
        print(f'Warning: Financial metrics failed for {model_name}: {e}')
        fin_metrics = {}
    
    return predictions, targets, {**ml_metrics, **fin_metrics}


def train_price_model(model_key, epochs=None, enable_physics=True):
    """Train a price prediction model."""
    print(f'\n{"="*60}')
    print(f'Training: {model_key}')
    print(f'{"="*60}')
    
    epochs = epochs or research_cfg.epochs
    
    model = registry.create_model(
        model_type=model_key,
        input_dim=input_dim,
        hidden_dim=research_cfg.hidden_dim,
        num_layers=research_cfg.num_layers,
        dropout=research_cfg.dropout
    )
    
    if model is None:
        print(f'Failed to create model: {model_key}')
        return None
    
    print(f'Model: {model.__class__.__name__}')
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
    
    trainer = Trainer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        test_loader=test_loader,
        device=device,
        research_mode=True
    )
    
    history = {'epochs': [], 'train_loss': [], 'val_loss': []}
    best_val_loss = float('inf')
    
    pbar = tqdm(range(1, epochs + 1), desc=model_key)
    for epoch in pbar:
        train_loss, _ = trainer.train_epoch(enable_physics=enable_physics)
        val_loss, _ = trainer.validate_epoch(enable_physics=enable_physics)
        
        history['epochs'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            trainer.save_checkpoint(epoch=epoch, val_loss=val_loss, 
                                    is_best=True, model_name=model_key)
        
        pbar.set_postfix({'train': f'{train_loss:.4f}', 'val': f'{val_loss:.4f}'})
    
    predictions, targets, metrics = evaluate_model(trainer.model, test_loader, model_key)
    
    print(f'Test Metrics:')
    print(f'  RMSE: {metrics.get("rmse", "N/A"):.4f}')
    print(f'  MAE: {metrics.get("mae", "N/A"):.4f}')
    print(f'  Directional Accuracy: {metrics.get("directional_accuracy", "N/A"):.2f}%')
    print(f'  Sharpe Ratio: {metrics.get("sharpe_ratio", "N/A"):.3f}')
    
    with open(results_dir / f'{model_key}_results.json', 'w') as f:
        json.dump({
            'model': model_key,
            'history': history,
            'metrics': {k: float(v) if isinstance(v, (np.floating, float)) else v 
                       for k, v in metrics.items()},
            'training_completed': True
        }, f, indent=2)
    
    return {
        'history': history,
        'metrics': metrics,
        'predictions': predictions,
        'targets': targets
    }


def train_volatility_model(model_key, dataset, epochs=30):
    """Train a volatility forecasting model."""
    if dataset is None or not HAS_VOL_TRAINER:
        print(f'Skipping {model_key} - volatility trainer not available')
        return None
    
    print(f'\n{"="*60}')
    print(f'Training Volatility Model: {model_key}')
    print(f'{"="*60}')
    
    model = registry.create_model(
        model_key,
        input_dim=dataset.X_train.shape[2],
        hidden_dim=64,
        num_layers=2,
        dropout=0.1
    )
    
    if model is None:
        print(f'Failed to create model: {model_key}')
        return None
    
    print(f'Model: {model.__class__.__name__}')
    print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
    
    trainer = VolatilityTrainer(
        model,
        learning_rate=1e-3,
        batch_size=64,
        max_epochs=epochs,
        patience=10,
        device=device,
        checkpoint_dir=results_dir
    )
    
    enable_physics = 'pinn' in model_key or 'heston' in model_key
    history = trainer.fit(dataset, enable_physics=enable_physics, verbose=True)
    
    # Get metrics from history
    metrics = {
        'final_train_loss': history.get('train_loss', [0])[-1] if history.get('train_loss') else 0,
        'final_val_loss': history.get('val_loss', [0])[-1] if history.get('val_loss') else 0,
        'best_val_loss': min(history.get('val_loss', [float('inf')])) if history.get('val_loss') else float('inf'),
    }
    
    print(f'Volatility Metrics:')
    print(f'  Final Train Loss: {metrics["final_train_loss"]:.6f}')
    print(f'  Final Val Loss: {metrics["final_val_loss"]:.6f}')
    print(f'  Best Val Loss: {metrics["best_val_loss"]:.6f}')
    
    with open(results_dir / f'{model_key}_results.json', 'w') as f:
        json.dump({
            'model': model_key,
            'history': {k: [float(x) for x in v] if isinstance(v, list) else v 
                       for k, v in history.items()},
            'metrics': metrics,
            'training_completed': True
        }, f, indent=2)
    
    return {
        'history': history,
        'metrics': metrics
    }

print('Training functions defined!')

In [ ]:
#@title 7. Train Baseline Models (5)
baseline_results = {}

for model_key in BASELINE_MODELS:
    try:
        result = train_price_model(model_key, enable_physics=False)
        if result:
            baseline_results[model_key] = result
    except Exception as e:
        print(f'Error training {model_key}: {e}')
        import traceback
        traceback.print_exc()

print(f'\nCompleted training {len(baseline_results)}/{len(BASELINE_MODELS)} baseline models')

In [ ]:
#@title 8. Train PINN Models (6)
pinn_results = {}

for model_key in PINN_MODELS:
    try:
        result = train_price_model(model_key, enable_physics=True)
        if result:
            pinn_results[model_key] = result
    except Exception as e:
        print(f'Error training {model_key}: {e}')
        import traceback
        traceback.print_exc()

print(f'\nCompleted training {len(pinn_results)}/{len(PINN_MODELS)} PINN models')

In [ ]:
#@title 9. Train Advanced PINN Models (3) - Including Spectral
advanced_results = {}

for model_key in ADVANCED_PINN_MODELS:
    try:
        result = train_price_model(model_key, enable_physics=True)
        if result:
            advanced_results[model_key] = result
    except Exception as e:
        print(f'Error training {model_key}: {e}')
        import traceback
        traceback.print_exc()

print(f'\nCompleted training {len(advanced_results)}/{len(ADVANCED_PINN_MODELS)} advanced PINN models')

In [ ]:
#@title 10. Train Volatility Models (6)
volatility_results = {}

for model_key in VOLATILITY_MODELS:
    try:
        result = train_volatility_model(model_key, vol_dataset, epochs=30)
        if result:
            volatility_results[model_key] = result
    except Exception as e:
        print(f'Error training {model_key}: {e}')
        import traceback
        traceback.print_exc()

print(f'\nCompleted training {len(volatility_results)}/{len(VOLATILITY_MODELS)} volatility models')

In [ ]:
#@title 11. Combine All Results
price_results = {**baseline_results, **pinn_results, **advanced_results}
all_results = {**price_results, **volatility_results}

print(f'=== TRAINING SUMMARY ===')
print(f'Total models trained: {len(all_results)}')
print(f'  - Baseline: {len(baseline_results)}')
print(f'  - PINN: {len(pinn_results)}')
print(f'  - Advanced PINN: {len(advanced_results)}')
print(f'  - Volatility: {len(volatility_results)}')
print(f'\nAll models: {list(all_results.keys())}')

In [ ]:
#@title 12. Generate Price Models Metrics Summary
summary_rows = []

for model_key, result in price_results.items():
    metrics = result['metrics']
    
    if model_key in BASELINE_MODELS:
        model_type = 'Baseline'
    elif model_key in PINN_MODELS:
        model_type = 'PINN'
    else:
        model_type = 'Advanced PINN'
    
    summary_rows.append({
        'Model': model_key,
        'Type': model_type,
        'RMSE': metrics.get('rmse'),
        'MAE': metrics.get('mae'),
        'MAPE': metrics.get('mape'),
        'R2': metrics.get('r2'),
        'Dir. Acc. (%)': metrics.get('directional_accuracy'),
        'Sharpe': metrics.get('sharpe_ratio'),
        'Sortino': metrics.get('sortino_ratio'),
        'Max DD (%)': metrics.get('max_drawdown', 0) * 100 if metrics.get('max_drawdown') else None
    })

price_summary_df = pd.DataFrame(summary_rows)
price_summary_df = price_summary_df.sort_values('RMSE')

print('\n' + '='*80)
print('PRICE PREDICTION MODEL PERFORMANCE SUMMARY')
print('='*80)
display(price_summary_df.round(4))

price_summary_df.to_csv(results_dir / 'price_model_summary.csv', index=False)
print(f'Summary saved to {results_dir / "price_model_summary.csv"}')

In [ ]:
#@title 13. Generate Volatility Models Metrics Summary
vol_summary_rows = []

for model_key, result in volatility_results.items():
    metrics = result['metrics']
    
    is_physics = 'pinn' in model_key or 'heston' in model_key
    
    vol_summary_rows.append({
        'Model': model_key,
        'Type': 'Physics-Informed' if is_physics else 'Baseline',
        'Final Train Loss': metrics.get('final_train_loss'),
        'Final Val Loss': metrics.get('final_val_loss'),
        'Best Val Loss': metrics.get('best_val_loss'),
    })

vol_summary_df = pd.DataFrame(vol_summary_rows)
vol_summary_df = vol_summary_df.sort_values('Best Val Loss')

print('\n' + '='*80)
print('VOLATILITY MODEL PERFORMANCE SUMMARY')
print('='*80)
display(vol_summary_df.round(6))

vol_summary_df.to_csv(results_dir / 'volatility_model_summary.csv', index=False)
print(f'Summary saved to {results_dir / "volatility_model_summary.csv"}')

In [ ]:
#@title 14. Plot Training Curves - Price Models
n_models = len(price_results)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten() if n_models > 1 else [axes]

for idx, (model_key, result) in enumerate(price_results.items()):
    ax = axes[idx]
    history = result['history']
    
    ax.plot(history['epochs'], history['train_loss'], label='Train', linewidth=2)
    ax.plot(history['epochs'], history['val_loss'], label='Val', linewidth=2, linestyle='--')
    
    ax.set_title(f'{model_key}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Training Curves - Price Prediction Models', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(results_dir / 'price_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Training curves saved to {results_dir / "price_training_curves.png"}')

In [ ]:
#@title 15. Plot Training Curves - Volatility Models
if volatility_results:
    n_vol = len(volatility_results)
    n_cols_vol = min(3, n_vol)
    n_rows_vol = (n_vol + n_cols_vol - 1) // n_cols_vol
    
    fig, axes = plt.subplots(n_rows_vol, n_cols_vol, figsize=(15, 4*n_rows_vol))
    if n_vol > 1:
        axes = axes.flatten()
    else:
        axes = [axes]
    
    for idx, (model_key, result) in enumerate(volatility_results.items()):
        ax = axes[idx]
        history = result['history']
        
        if 'train_loss' in history and 'val_loss' in history:
            epochs = range(1, len(history['train_loss']) + 1)
            ax.plot(epochs, history['train_loss'], label='Train', linewidth=2)
            ax.plot(epochs, history['val_loss'], label='Val', linewidth=2, linestyle='--')
        
        ax.set_title(f'{model_key}', fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend()
        ax.grid(True, alpha=0.3)
    
    for idx in range(n_vol, len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Training Curves - Volatility Models', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(results_dir / 'volatility_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'Volatility training curves saved to {results_dir / "volatility_training_curves.png"}')
else:
    print('No volatility models trained')

In [ ]:
#@title 16. Plot Metrics Comparison - All Price Models
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = []
for model in price_summary_df['Model']:
    if model in BASELINE_MODELS:
        colors.append('#2E86AB')
    elif model in PINN_MODELS:
        colors.append('#A23B72')
    else:
        colors.append('#F18F01')

ax1 = axes[0, 0]
ax1.barh(price_summary_df['Model'], price_summary_df['RMSE'], color=colors)
ax1.set_xlabel('RMSE (lower is better)')
ax1.set_title('RMSE by Model')
ax1.invert_yaxis()

ax2 = axes[0, 1]
ax2.barh(price_summary_df['Model'], price_summary_df['Dir. Acc. (%)'], color=colors)
ax2.set_xlabel('Directional Accuracy %')
ax2.set_title('Directional Accuracy by Model')
ax2.invert_yaxis()
ax2.axvline(x=50, color='red', linestyle='--', alpha=0.5, label='Random (50%)')

ax3 = axes[1, 0]
sharpe_sorted = price_summary_df.sort_values('Sharpe', ascending=True)
colors_sharpe = []
for model in sharpe_sorted['Model']:
    if model in BASELINE_MODELS:
        colors_sharpe.append('#2E86AB')
    elif model in PINN_MODELS:
        colors_sharpe.append('#A23B72')
    else:
        colors_sharpe.append('#F18F01')

ax3.barh(sharpe_sorted['Model'], sharpe_sorted['Sharpe'], color=colors_sharpe)
ax3.set_xlabel('Sharpe Ratio (higher is better)')
ax3.set_title('Sharpe Ratio by Model')
ax3.invert_yaxis()
ax3.axvline(x=0, color='red', linestyle='--', alpha=0.5)

ax4 = axes[1, 1]
type_metrics = price_summary_df.groupby('Type').agg({
    'RMSE': 'mean',
    'Dir. Acc. (%)': 'mean',
    'Sharpe': 'mean'
}).round(4)

x = np.arange(len(type_metrics))
width = 0.25

rmse_norm = type_metrics['RMSE'] / type_metrics['RMSE'].max()
da_norm = type_metrics['Dir. Acc. (%)'] / 100
sharpe_norm = (type_metrics['Sharpe'] - type_metrics['Sharpe'].min()) / \
              (type_metrics['Sharpe'].max() - type_metrics['Sharpe'].min() + 1e-6)

ax4.bar(x - width, rmse_norm, width, label='RMSE (norm)', color='#2E86AB')
ax4.bar(x, da_norm, width, label='Dir. Acc. (norm)', color='#A23B72')
ax4.bar(x + width, sharpe_norm, width, label='Sharpe (norm)', color='#F18F01')
ax4.set_xticks(x)
ax4.set_xticklabels(type_metrics.index)
ax4.set_ylabel('Normalized Score')
ax4.set_title('Average Metrics by Model Type')
ax4.legend()

plt.tight_layout()
plt.savefig(results_dir / 'metrics_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Metrics comparison saved to {results_dir / "metrics_comparison.png"}')

In [ ]:
#@title 17. Plot Predictions vs Actuals (Best Models)
if baseline_results and pinn_results:
    best_baseline = min([m for m in baseline_results.keys()], 
                        key=lambda m: baseline_results[m]['metrics'].get('rmse', float('inf')))
    best_pinn = min([m for m in pinn_results.keys()], 
                    key=lambda m: pinn_results[m]['metrics'].get('rmse', float('inf')))
    
    print(f'Best Baseline: {best_baseline}')
    print(f'Best PINN: {best_pinn}')
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    ax1 = axes[0, 0]
    result = baseline_results[best_baseline]
    preds, targs = result['predictions'][:200], result['targets'][:200]
    ax1.plot(targs, label='Actual', alpha=0.7)
    ax1.plot(preds, label='Predicted', alpha=0.7)
    ax1.set_title(f'{best_baseline} - Predictions vs Actual')
    ax1.set_xlabel('Sample')
    ax1.set_ylabel('Log Return')
    ax1.legend()
    
    ax2 = axes[0, 1]
    ax2.scatter(result['targets'], result['predictions'], alpha=0.3, s=10)
    ax2.plot([min(result['targets']), max(result['targets'])], 
             [min(result['targets']), max(result['targets'])], 'r--', label='Perfect')
    ax2.set_xlabel('Actual')
    ax2.set_ylabel('Predicted')
    ax2.set_title(f'{best_baseline} - Scatter Plot')
    ax2.legend()
    
    ax3 = axes[1, 0]
    result = pinn_results[best_pinn]
    preds, targs = result['predictions'][:200], result['targets'][:200]
    ax3.plot(targs, label='Actual', alpha=0.7)
    ax3.plot(preds, label='Predicted', alpha=0.7)
    ax3.set_title(f'{best_pinn} - Predictions vs Actual')
    ax3.set_xlabel('Sample')
    ax3.set_ylabel('Log Return')
    ax3.legend()
    
    ax4 = axes[1, 1]
    ax4.scatter(result['targets'], result['predictions'], alpha=0.3, s=10)
    ax4.plot([min(result['targets']), max(result['targets'])], 
             [min(result['targets']), max(result['targets'])], 'r--', label='Perfect')
    ax4.set_xlabel('Actual')
    ax4.set_ylabel('Predicted')
    ax4.set_title(f'{best_pinn} - Scatter Plot')
    ax4.legend()
    
    plt.tight_layout()
    plt.savefig(results_dir / 'predictions_vs_actual.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
#@title 18. Final Comprehensive Report
print('='*80)
print('FINAL TRAINING REPORT - ALL MODELS')
print('='*80)
print(f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Device: {device}')
print(f'Research Config: epochs={research_cfg.epochs}, batch_size={research_cfg.batch_size}')

print(f'\n=== TRAINING SUMMARY ===')
print(f'Total models trained: {len(all_results)}')
print(f'  - Baseline: {len(baseline_results)}/{len(BASELINE_MODELS)}')
print(f'  - PINN: {len(pinn_results)}/{len(PINN_MODELS)}')
print(f'  - Advanced PINN (incl. Spectral): {len(advanced_results)}/{len(ADVANCED_PINN_MODELS)}')
print(f'  - Volatility: {len(volatility_results)}/{len(VOLATILITY_MODELS)}')

print('\n' + '-'*40)
print('BEST PRICE PREDICTION MODELS')
print('-'*40)

if len(price_summary_df) > 0:
    best_rmse = price_summary_df.loc[price_summary_df['RMSE'].idxmin()]
    print(f'Best RMSE: {best_rmse["Model"]} ({best_rmse["RMSE"]:.4f})')
    
    best_da = price_summary_df.loc[price_summary_df['Dir. Acc. (%)'].idxmax()]
    print(f'Best Directional Accuracy: {best_da["Model"]} ({best_da["Dir. Acc. (%)"]:.2f}%)')
    
    best_sharpe = price_summary_df.loc[price_summary_df['Sharpe'].idxmax()]
    print(f'Best Sharpe Ratio: {best_sharpe["Model"]} ({best_sharpe["Sharpe"]:.3f})')

print('\n' + '-'*40)
print('BEST VOLATILITY MODELS')
print('-'*40)

if len(vol_summary_df) > 0:
    best_vol = vol_summary_df.loc[vol_summary_df['Best Val Loss'].idxmin()]
    print(f'Best Val Loss: {best_vol["Model"]} ({best_vol["Best Val Loss"]:.6f})')

print('\n' + '-'*40)
print('PINN vs BASELINE COMPARISON (Price Models)')
print('-'*40)

if len(price_summary_df) > 0:
    baseline_avg = price_summary_df[price_summary_df['Type'] == 'Baseline'].mean(numeric_only=True)
    pinn_avg = price_summary_df[price_summary_df['Type'] == 'PINN'].mean(numeric_only=True)
    
    if not baseline_avg.empty and not pinn_avg.empty:
        print(f'Average RMSE:')
        print(f'  Baseline: {baseline_avg["RMSE"]:.4f}')
        print(f'  PINN: {pinn_avg["RMSE"]:.4f}')
        improvement = (1 - pinn_avg["RMSE"]/baseline_avg["RMSE"])*100
        print(f'  Improvement: {improvement:.2f}%')
        
        print(f'\nAverage Directional Accuracy:')
        print(f'  Baseline: {baseline_avg["Dir. Acc. (%)"]:.2f}%')
        print(f'  PINN: {pinn_avg["Dir. Acc. (%)"]:.2f}%')
        
        print(f'\nAverage Sharpe Ratio:')
        print(f'  Baseline: {baseline_avg["Sharpe"]:.3f}')
        print(f'  PINN: {pinn_avg["Sharpe"]:.3f}')

print('\n' + '='*80)
print(f'Results saved to: {results_dir}')
print('Files generated:')
print('  - price_model_summary.csv')
print('  - volatility_model_summary.csv')
print('  - price_training_curves.png')
print('  - volatility_training_curves.png')
print('  - metrics_comparison.png')
print('  - predictions_vs_actual.png')
print('  - {model}_results.json for each model')
print('='*80)